In [1]:
import pandas as pd
df=pd.read_csv('myntra_products_catalog.csv')
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


# **Target Brands**

In [2]:
target_brands = ['H&M','U.S. Polo Assn.','GAP',"Levi's",'Tommy Hilfiger']
df[df['ProductBrand'].isin(target_brands)]['ProductBrand'].value_counts()

,count
ProductBrand,
GAP,216
Tommy Hilfiger,78
U.S. Polo Assn.,47


# **Search for H&M and Levi's under different spellings**

In [3]:
df[df['ProductBrand'].str.contains('h&m|Levi',case=False,na=False)]['ProductBrand'].value_counts()

,count
ProductBrand,
Levis,67
Denizen From Levis,2


# **Filter the dataset**

In [4]:
target_brands=['GAP','Tommy Hilfiger','Levis','U.S. Polo Assn.']
brand_df=df[df['ProductBrand'].isin(target_brands)].copy()
brand_df.shape

(408, 8)

# **missing values**

In [5]:
brand_df.isna().sum()

,0
ProductID,0
ProductName,0
ProductBrand,0
Gender,0
Price (INR),0
NumImages,0
Description,0
PrimaryColor,205


# **Fill missing color values**

In [6]:
brand_df['PrimaryColor']=brand_df['PrimaryColor'].fillna('Unknown')
brand_df['PrimaryColor'].isna().sum()

np.int64(0)

## Note on Return Data

No public dataset exists with real return data tied to named fashion brands
(companies do not publicly release this). To enable return-pattern analysis,
a `simulated_returned` column was generated using documented, realistic
assumptions (higher-priced items and certain categories have a higher
simulated return likelihood). This column is clearly synthetic and does not
reflect real return data from these brands.

# **price cutoff for "expensive" items**

In [7]:
price_threshold=brand_df['Price (INR)'].quantile(0.75)
price_threshold

np.float64(1999.0)

# **simulated returns column**

In [8]:
import numpy as np

np.random.seed(42)

def calculate_return_chance(row):
  chance=0.10
  if row['Price (INR)']>price_threshold:
    chance+=0.10
  name_lower=row['ProductName'].lower()
  if any(word in name_lower for word in ['jeans','dress','fit','skinny']):
    chance+=0.08
  return min(chance,0.35)
brand_df['return_possibility']=brand_df.apply(calculate_return_chance,axis=1)
brand_df['simulated_returned']=np.random.binomial(1,brand_df['return_possibility'])
brand_df[['ProductName','Price (INR)','return_possibility','simulated_returned']].head(20)

,ProductName,Price (INR),return_possibility,simulated_returned
1977,Levis Men Blue 511 Slim Fit Mid-Rise Clean Loo...,2039,0.28,0
2003,Levis Men Grey Slim Fit Mid-Rise Clean Look St...,1649,0.18,1
2095,Levis Men White 511 Slim Fit Mid-Rise Clean Lo...,1799,0.18,0
2326,Levis Men Grey 65504 Skinny Straight Fit Mid-R...,1799,0.18,0
2381,Levis Men Blue 511 Slim Fit Mid-Rise Clean Loo...,2039,0.28,0
2720,Levis Men Blue 511 Slim Fit Mid-Rise Clean Loo...,2999,0.28,0
2835,Levis Men Blue 511 Slim Fit Mid-Rise Clean Loo...,2199,0.28,0
3484,Levis Men Brown Solid Biker Jacket,6999,0.20,1
3823,Levis Men Grey Melange Solid Water Resistant T...,5999,0.20,0
3878,Levis Men Olive Green & Brown Printed Water Re...,2999,0.20,0


# **overall simulated return rate**

In [9]:
brand_df['simulated_returned'].mean()

np.float64(0.13725490196078433)

# **load it into SQLite**

In [10]:
import sqlite3
conn=sqlite3.connect('fashion_brands.db')
brand_df.to_sql('products',conn,if_exists='replace',index=False)

408

# **load data**

In [11]:
query="SELECT * FROM products LIMIT 5"
pd.read_sql(query,conn)

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor,return_possibility,simulated_returned
0,10038559,Levis Men Blue 511 Slim Fit Mid-Rise Clean Loo...,Levis,Men,2039,7,"Blue medium wash 5-pocket mid-rise jeans, clea...",Blue,0.28,0
1,10038513,Levis Men Grey Slim Fit Mid-Rise Clean Look St...,Levis,Men,1649,7,"Grey light wash 5-pocket mid-rise jeans, clean...",Grey,0.18,1
2,10038501,Levis Men White 511 Slim Fit Mid-Rise Clean Lo...,Levis,Men,1799,7,"White light wash 5-pocket mid-rise jeans, clea...",White,0.18,0
3,10038557,Levis Men Grey 65504 Skinny Straight Fit Mid-R...,Levis,Men,1799,7,"Grey light wash 5-pocket mid-rise jeans, clean...",Grey,0.18,0
4,10038517,Levis Men Blue 511 Slim Fit Mid-Rise Clean Loo...,Levis,Men,2039,7,"Blue dark wash 5-pocket mid-rise jeans, clean ...",Blue,0.28,0


# **Average price by brand**

In [12]:
query = """
SELECT ProductBrand , ROUND(AVG("Price (INR)"),2) AS avg_price , count(*) as num_product
FROM products
GROUP BY ProductBrand
ORDER BY avg_price DESC
"""
pd.read_sql(query,conn)

,ProductBrand,avg_price,num_product
0,Levis,3441.84,67
1,GAP,1404.46,216
2,Tommy Hilfiger,1371.81,78
3,U.S. Polo Assn.,761.72,47


# **Check average price by brand AND product type**

In [13]:
query= """
SELECT ProductBrand,
 CASE
  WHEN ProductName LIKE '%Jacket%' THEN 'Jacket'
  WHEN ProductName LIKE '%Jeans%' THEN 'Jeans'
  WHEN ProductName LIKE '%Shirt%' THEN 'Shirt'
  WHEN ProductName LIKE '%T-Shirt%' OR ProductName LIKE '%Tshirt%' THEN 'T-Shirt'
  ELSE 'Others'
END AS Product_type,
ROUND(AVG("Price (INR)"),2) AS avg_price , count(*) as num_product
FROM Products
GROUP BY ProductBrand , Product_type
ORDER BY ProductBrand , avg_price DESC
"""
pd.read_sql(query,conn)



,ProductBrand,Product_type,avg_price,num_product
0,GAP,Jacket,1991.86,7
1,GAP,Jeans,1621.22,9
2,GAP,Others,1553.18,67
3,GAP,Shirt,1283.96,133
4,Levis,Jacket,5332.33,3
5,Levis,Others,3520.18,51
6,Levis,Shirt,3425.67,6
7,Levis,Jeans,2074.71,7
8,Tommy Hilfiger,Jeans,2303.00,5
9,Tommy Hilfiger,Shirt,1329.00,22


Levis' higher average price isn't driven by its core product — jeans — which are actually priced lower than Tommy Hilfiger's jeans. Instead, Levis' overall premium comes from higher-priced items outside its signature category, like jackets and other apparel. This suggests Levis' brand pricing strategy leans on premium positioning across its broader catalog, not on charging more for the product it's best known for.

# **Simulated return rate by brand**

In [14]:
query = """
SELECT ProductBrand , ROUND(AVG(simulated_returned)*100,1) as return_rate , count(*) as num_product
FROM Products
GROUP BY ProductBrand
ORDER BY return_rate DESC
"""
pd.read_sql(query,conn)

,ProductBrand,return_rate,num_product
0,Levis,22.4,67
1,GAP,13.4,216
2,Tommy Hilfiger,12.8,78
3,U.S. Polo Assn.,4.3,47


# **Install GoogleAI Library**

In [15]:
!pip install -q google-generativeai

# **Set up API key in the code**

In [16]:
import google.generativeai as genai
genai.configure(api_key="API_KEY_HERE")

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### **Removing the old package **

In [17]:
!pip uninstall google-generativeai -y -q
!pip install -U google-genai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


# **connect**

In [18]:
from google import genai
client = genai.Client(api_key="API_KEY_HERE")

# **QUESTION TO GEMINI**

In [19]:
response = client.models.generate_content(model = "gemini-3.1-flash-lite", contents= "say hello and confirm this is working.")
print(response.text)


Hello! I can confirm that this is working perfectly. How can I help you today?


# **SQL QUERY TROUGH AI**

In [20]:
schema_info = """
Table Name = products
columns: ProductID , ProductName , ProductBrand , Gender , "Price (INR)" , NumImages , Description , PrimaryColor , return_possibility , simulated_returned
"""
question = "Which brand has the highest average price?"
prompt = f"""
you are a sql expert . give this table schema : {schema_info}
write a sql lite query to answer this question :{question}
only return the sql query , nothing else .
"""
response= client.models.generate_content(model= "gemini-3.1-flash-lite",contents=prompt)
print(response.text)


SELECT ProductBrand FROM products GROUP BY ProductBrand ORDER BY AVG("Price (INR)") DESC LIMIT 1;


# **AI QUERY**

In [21]:
ai_query = 'SELECT ProductBrand FROM products GROUP BY ProductBrand ORDER BY AVG("Price (INR)") DESC LIMIT 1;'
pd.read_sql(ai_query,conn)

,ProductBrand
0,Levis


# **improved query**

In [22]:
improved_query = """
SELECT ProductBrand , ROUND(AVG("Price (INR)"),2) AS avg_price , count(*) as num_product
FROM products
GROUP BY ProductBrand
ORDER BY avg_price DESC
LIMIT 1
"""
pd.read_sql(improved_query,conn)

,ProductBrand,avg_price,num_product
0,Levis,3441.84,67


# **another sql ai query**

In [23]:
question2= "what percentage of Tommy Hilfiger products are simulated as returned?"
prompt2= f"""
you are a sql expert . given this table schema : {schema_info}
write a sql lite query to answer this question : {question2}
only return the sql query , nothing else.
"""
response2= client.models.generate_content(model="gemini-3.1-flash-lite",contents=prompt2)
print(response2.text)

SELECT (SUM(simulated_returned) * 100.0 / COUNT(*)) FROM products WHERE ProductBrand = 'Tommy Hilfiger';


## **check against our earliest result**

In [24]:
query2="SELECT (SUM(simulated_returned) * 100.0) / COUNT(*) FROM products WHERE ProductBrand = 'Tommy Hilfiger';"
pd.read_sql(query2,conn)

,(SUM(simulated_returned) * 100.0) / COUNT(*)
0,12.820513


# **testing a real weak spot**

In [25]:
question3="how many H&M products are in this dataset?"
prompt3= f"""
you are a sql expert . given this table schema : {schema_info}
write a sql lite query to answer this question : {question3}
only return the sql query , nothing else
"""
response3 = client.models.generate_content(model="gemini-3.1-flash-lite",contents=prompt3)
print(response3.text)

SELECT COUNT(*) FROM products WHERE ProductBrand = 'H&M';


# **runnin it **

In [26]:
query3="SELECT COUNT(*) FROM products WHERE ProductBrand = 'H&M';"
pd.read_sql(query3,conn)

,COUNT(*)
0,0


# **ambiguous one**

In [27]:
question4="what is the average price of items that are likely to be returned ?"
prompt4=f"""
you are a sql expert . given this table schema : {schema_info}
write a sql query for this question : {question4}
only return the sql query , nothing else.
"""
response4=client.models.generate_content(model="gemini-3.1-flash-lite",contents=prompt4)
print(response4.text)

SELECT AVG("Price (INR)") FROM products WHERE return_possibility = 1;


# **correct version**

In [28]:
query4_correct = 'SELECT ROUND(AVG("Price (INR)"), 2) AS avg_price FROM products WHERE simulated_returned = 1;'
pd.read_sql(query4_correct, conn)


,avg_price
0,2103.77


# **product_type as a permanent column, then export**

In [29]:
def get_product_type(name):
  name_lower = name.lower()
  if 'jacket' in name_lower:
    return 'Jacket'
  elif 'jeans' in name_lower:
    return 'Jeans'
  elif 't-shirt' in name_lower or 'tshirt' in name_lower:
    return 'T-shirt'
  elif 'shirt' in name_lower:
    return 'Shirt'
  else:
    return 'Others'
brand_df['Product_type']=brand_df['ProductName'].apply(get_product_type)
brand_df.to_csv('fashion_brands_final.csv',index=False)
print('Saved!')

Saved!
